# 01. Engineering Data Reality Check
**Engineering Data Processing with Python** | one-day course | about 30 minutes

> **TRAINER / SOLUTION VERSION.** Every blank is filled in and the notebook has been run.


In [1]:
# @title Setup: run this cell first (it loads the course data)
# ===================== SETUP: run this cell first (click it, then Shift+Enter) =====================
# It makes sure the course data is available in this Colab session and loads the tools we need.
import os, zipfile, urllib.request
from pathlib import Path

DATA_ZIP_URL = "https://raw.githubusercontent.com/buildyourai/ei-python-data-processing/main/data.zip"   # course data (fictional / public)

def get_course_data():
    if Path("data").is_dir():
        print("Data folder found:", sorted(p.name for p in Path("data").iterdir())[:20])
        return
    if not Path("data.zip").exists() and DATA_ZIP_URL:
        print("Downloading course data...")
        urllib.request.urlretrieve(DATA_ZIP_URL, "data.zip")
    zip_path = Path("data.zip")
    if not zip_path.exists():
        try:
            from google.colab import files
            print("Choose data.zip from your computer (your trainer will tell you where it is).")
            uploaded = files.upload()
            zip_path = Path(next(iter(uploaded)))
        except ImportError:
            raise FileNotFoundError("Put data.zip or the data folder next to this notebook.")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(".")
    print("Data ready:", sorted(p.name for p in Path("data").iterdir()))

get_course_data()
Path("outputs").mkdir(exist_ok=True)

import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

def check(condition, ok="Looks right.", hint="Not quite yet. Re-read the instructions above the cell."):
    """Tiny self-check helper used throughout the course."""
    print(("PASS: " + ok) if condition else ("NOT YET: " + hint))

Data folder found: ['3dprinter.csv', 'asset_register.xlsx', 'boiler.json', 'capstone', 'checkpoints', 'concrete.csv', 'manifest.TXT', 'manifest.csv', 'sensor_readings.csv', 'steel.csv', 'thermal-profiles.xlsx', 'work_orders_cmms.csv', 'work_orders_legacy.xlsx']


## The point of this module

Most data problems in engineering exports are boring and predictable. Someone typed `???` in a cell. A sensor wrote
`-999` when it lost signal. A column name has a space at the end you cannot see. None of these are hard to fix.
The danger is not noticing them and reporting a number that is confidently wrong.

### Problems you will meet today

| Problem | Example | Why it hurts |
|---|---|---|
| Numbers stored as text | `"15 kW"`, `"€1,250.00"`, `"???"` | Column will not add up or average |
| Day/month order | `01/02/2026`: 1 Feb or 2 Jan? | Wrong month, no error |
| Mixed units | °F from one logger, °C from another | Plausible but wrong |
| "No reading" codes | `-999`, `9999` | Wrecks averages, fires false alarms |
| Spaces and capitals | `"CNC-001 "` vs `"cnc-001"` | Records fail to match |
| Duplicates | Same record exported twice | Counts and costs inflated |
| Many spellings | `CM`, `Corrective`, `breakdown` | Totals split three ways |

### Five rules for repeatable work

1. Never edit the raw file by hand. Fix it in code, so the fix is recorded.
2. Every step lives in the notebook.
3. Before trusting a result: **Runtime > Restart session and run all**.
4. Save outputs to a separate folder.
5. Count rows before and after every step.

## 1. Why won't this column average?

The 3D printer test data again. Run and read.

In [2]:
printer = pd.read_csv("data/3dprinter.csv")
printer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51 entries, 0 to 50
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   layer_height        51 non-null     float64
 1   wall_thickness      51 non-null     int64  
 2   infill_density      51 non-null     int64  
 3   infill_pattern      51 non-null     object 
 4   nozzle_temperature  51 non-null     int64  
 5   bed_temperature     51 non-null     int64  
 6   print_speed         51 non-null     int64  
 7   material            51 non-null     object 
 8   fan_speed           51 non-null     int64  
 9   roughness           51 non-null     object 
 10  tension_strenght    50 non-null     float64
 11  elongation          50 non-null     float64
dtypes: float64(3), int64(6), object(3)
memory usage: 4.9+ KB


`roughness` shows as `object`. In pandas that means **text**. One value that is not a number turns the whole column
into text. This cell finds it:

In [3]:
as_number = pd.to_numeric(printer["roughness"], errors="coerce")   # anything that isn't a number becomes NaN (missing)
printer[as_number.isna()]

,layer_height,wall_thickness,infill_density,infill_pattern,nozzle_temperature,bed_temperature,print_speed,material,fan_speed,roughness,tension_strenght,elongation
50,0.1,5,40,grid,210,70,50,pla,120,???,NaN,NaN


The last test has `???` for roughness and blanks for strength and elongation. Also notice the column
`tension_strenght` is misspelt. Typos in column names break code in confusing ways.

The fix for roughness: keep the row, turn `???` into "missing". Do not delete it and do not guess a value.

In [4]:
printer["roughness"] = pd.to_numeric(printer["roughness"], errors="coerce")
printer.groupby("material")["roughness"].mean()      # now it works

material
abs    193.44
pla    147.72
Name: roughness, dtype: float64

### Your turn 1.1
Fix the misspelt column name. Replace the blank with the correct spelling, `tension_strength`.

In [5]:
printer = printer.rename(columns={"tension_strenght": "tension_strength"})
printer.columns.tolist()

['layer_height',
 'wall_thickness',
 'infill_density',
 'infill_pattern',
 'nozzle_temperature',
 'bed_temperature',
 'print_speed',
 'material',
 'fan_speed',
 'roughness',
 'tension_strength',
 'elongation']

In [6]:
check("tension_strength" in printer.columns, "Renamed.", "Check the spelling: tension_strength")

PASS: Renamed.


## 2. Hidden spaces in column names

Concrete compressive strength data. The numbers look sensible. The column names do not. Printing them with `repr()`
shows hidden spaces.

In [7]:
concrete = pd.read_csv("data/concrete.csv")
for name in concrete.columns:
    print(repr(name))

'Cement (component 1)(kg in a m^3 mixture)'
'Blast Furnace Slag (component 2)(kg in a m^3 mixture)'
'Fly Ash (component 3)(kg in a m^3 mixture)'
'Water  (component 4)(kg in a m^3 mixture)'
'Superplasticizer (component 5)(kg in a m^3 mixture)'
'Coarse Aggregate  (component 6)(kg in a m^3 mixture)'
'Fine Aggregate (component 7)(kg in a m^3 mixture)'
'Age (day)'
'Concrete compressive strength(MPa, megapascals) '


`'Water  (component 4)...'` has **two** spaces, and the strength column ends with a space. Type the name as you see it
in Excel and it will not match. Easiest fix: give every column a short, clear name with its unit.

### Your turn 1.2
Two names are missing. Use `water_kg_m3` and `strength_mpa`.

In [8]:
concrete.columns = [
    "cement_kg_m3", "slag_kg_m3", "fly_ash_kg_m3", "water_kg_m3", "superplasticiser_kg_m3",
    "coarse_agg_kg_m3", "fine_agg_kg_m3", "age_days", "strength_mpa",
]
concrete.head()

,cement_kg_m3,slag_kg_m3,fly_ash_kg_m3,water_kg_m3,superplasticiser_kg_m3,coarse_agg_kg_m3,fine_agg_kg_m3,age_days,strength_mpa
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.986111
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.887366
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.269535
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.052780
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.296075


In [9]:
check(concrete.columns[3] == "water_kg_m3" and concrete.columns[-1] == "strength_mpa", "Readable names.",
      "Expected water_kg_m3 and strength_mpa.")

PASS: Readable names.


## 3. Duplicates and missing values

Two counts worth running on every new dataset. Example: count missing values in each column.

In [10]:
concrete.isna().sum()

cement_kg_m3              0
slag_kg_m3                0
fly_ash_kg_m3             0
water_kg_m3               0
superplasticiser_kg_m3    0
coarse_agg_kg_m3          0
fine_agg_kg_m3            0
age_days                  0
strength_mpa              0
dtype: int64

### Your turn 1.3
Now count duplicate rows. Choose from: `isna`, `duplicated`.

In [11]:
concrete.duplicated().sum()

np.int64(25)

25 rows are exact copies of other rows. Errors, or genuine repeat tests? Hold that thought for Module 04.

## 4. A one-line health check for any table

The next cell creates a helper, `profile(df)`, that summarises every column: type, missing values, number of
different values and an example. You do not need to read it, just run it.

In [12]:
# @title Helper: profile(df) gives a summary of every column (just run it)
def profile(df):
    return pd.DataFrame({
        "type": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "different_values": df.nunique(),
        "example": df.apply(lambda col: col.dropna().iloc[0] if col.notna().any() else None),
    })
print("profile() is ready")

profile() is ready


## 5. Meet the main dataset for today

**Kilbrack Engineering** is a fictional manufacturer with plants in Cork, Galway and Limerick. Its data is made up,
but the problems planted in it are the common ones. Here are its maintenance work orders.

In [13]:
wo = pd.read_csv("data/work_orders_cmms.csv")
print(wo.shape)
profile(wo)

(1016, 12)


,type,missing,different_values,example
wo_id,object,0,973,WO-25-00270
asset_id,object,0,159,PMP-005
wo_type,object,0,12,PM
priority,object,21,7,3
date_raised,object,0,967,23/06/2025 13:15
date_closed,object,41,955,24/06/2025 01:17
status,object,0,5,Closed
labour_hours,float64,36,32,2.5
parts_cost_eur,object,68,802,222.81
technician,object,0,32,Gráinne FitzGerald


### Your turn 1.4
Count the different spellings of work order **type**. The column is called `wo_type`.

In [14]:
wo["wo_type"].value_counts()

wo_type
PM             321
CM             139
PM              96
Preventive      91
pm              88
Corrective      54
cm              50
INSP            47
corrective      44
Breakdown       42
insp            24
Inspection      20
Name: count, dtype: int64

Twelve spellings of three types. Every column is text, including hours, cost and dates. And `wo_id` has fewer
different values than there are rows, so some work orders appear twice. That is Modules 02 to 04.

## 6. Discussion: what is "clean enough"?

There is no perfectly clean data, only data clean enough for a specific question.

The operations manager asks: **"Which type of asset costs us most in breakdown repairs?"**

With your neighbour: which columns must be right to answer that? Which problems from the table at the top would
change the answer? Which columns could you ignore?

<details><summary><b>Show a model answer</b></summary>

* **Must be right:** work order type (to pick out breakdowns), cost, asset ID (to find the asset type).
* **Problems that change the answer:** the 12 type spellings; cost stored as text with € and commas; duplicates
  inflating totals; asset IDs with spaces or lower case not matching the asset list.
* **Can ignore for this question:** technician, description, priority.
</details>

**Optional, with Gemini:** paste the output of `profile(wo)` into Gemini and ask the same question. Did it spot
anything you missed? Did it assume a column exists that does not?

Next: **02 Loading Engineering Datasets**.